# Module 2.1: Exploratory Data Analysis (EDA) on Industrial HVAC & Oven Logs
### 🇵🇱 Eksploracyjna Analiza Danych (EDA) Pieców Hutniczych i HVAC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m02_l01_eda_hvac_oven_logs.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Analyze industrial thermal profiles, identify heating/cooling hysteresis, and spot thermal sensor cross-correlations across oven zones.  
> 🇵🇱 **Streszczenie:** Badaj profile termiczne pieców, identyfikuj histerezę nagrzewania i chłodzenia oraz wyliczaj korelacje między strefami grzewczymi.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Multi-Zone Thermal Dynamics**: Industrial tunnel curing ovens and chemical reactors have distinct temperature zones (Preheat, Soak, Cure, Cool) coupled by thermal convection.
2. **Thermal Lag & Inertia**: Adjusting a burner duty cycle does not instantly heat the workpiece. The physical heat capacitance introduces a 30s-120s transport delay.
3. **Cross-Correlation**: Computing $R_{xy}(\tau) = \sum x(t) y(t+\tau)$ determines the exact physical transport lag between heating commands and thermocouple responses.


In [ ]:
# 1. Simulate 4-Zone Industrial Tunnel Oven Telemetry
np.random.seed(42)
n_seconds = 1800  # 30 minutes of continuous operation at 1 Hz
t = np.arange(n_seconds)

# Heater duty cycle (actuator input with pulse changes)
heater_duty = 50.0 + 20.0 * np.sin(2 * np.pi * t / 600) + np.random.normal(0, 1.5, n_seconds)

# Zones have thermal transport lags: Zone 1 (15s lag), Zone 2 (45s lag), Zone 3 (90s lag)
def simulate_zone(heater, lag_sec, gain, baseline, noise_std):
    delayed_heater = np.roll(heater, lag_sec)
    delayed_heater[:lag_sec] = heater[0]
    # Low-pass thermal filter (exponential moving average)
    alpha = 0.05
    zone_temp = np.zeros(len(heater))
    zone_temp[0] = baseline
    for i in range(1, len(heater)):
        target = baseline + gain * (delayed_heater[i] - 50.0)
        zone_temp[i] = (1 - alpha) * zone_temp[i-1] + alpha * target
    return zone_temp + np.random.normal(0, noise_std, len(heater))

zone1 = simulate_zone(heater_duty, lag_sec=15, gain=1.2, baseline=120.0, noise_std=0.4)
zone2 = simulate_zone(heater_duty, lag_sec=45, gain=1.8, baseline=180.0, noise_std=0.6)
zone3 = simulate_zone(heater_duty, lag_sec=90, gain=2.1, baseline=220.0, noise_std=0.8)

df_oven = pd.DataFrame({
    "time_sec": t, "heater_duty_pct": heater_duty,
    "zone1_preheat": zone1, "zone2_soak": zone2, "zone3_cure": zone3
})
print("✓ Simulated 30-minute 4-Zone Oven Telemetry Matrix.")


In [ ]:
# 2. Cross-Correlation Lag Analysis & Thermal Heatmap
from scipy.signal import correlate

# Calculate cross-correlation between heater duty and Zone 2 response
norm_heater = heater_duty - np.mean(heater_duty)
norm_zone2 = zone2 - np.mean(zone2)
corr = correlate(norm_zone2, norm_heater, mode='full')
lags = np.arange(-n_seconds + 1, n_seconds)

optimal_lag = lags[np.argmax(corr)]
print(f"Identified Physical Thermal Transport Lag: {optimal_lag} seconds.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cross-correlation curve
axes[0].plot(lags, corr, color='#0284c7', linewidth=1.8)
axes[0].axvline(optimal_lag, color='#ef4444', linestyle='--', label=f'Peak Lag = {optimal_lag}s')
axes[0].set_xlim(-50, 150)
axes[0].set_title("Cross-Correlation vs Time Lag (Duty Cycle to Zone 2)", fontweight='bold')
axes[0].set_xlabel("Lag [seconds]")
axes[0].set_ylabel("Correlation Coefficient")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Correlation Heatmap
sns.heatmap(df_oven.corr(), annot=True, fmt='.2f', cmap='Blues', ax=axes[1], cbar=False)
axes[1].set_title("Inter-Zone Sensor Cross-Correlation Matrix", fontweight='bold')

plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Visualize multi-zone oven temperature telemetry over a continuous 24-hour shift *(🇵🇱 Zwizualizuj profile temperatury stref pieca w trakcie 24-godzinnej zmiany roboczej)*
- [ ] **Task 2:** Calculate thermal cross-correlation matrix across all 12 heating zones *(🇵🇱 Oblicz macierz korelacji wzajemnej dla 12 stref grzewczych pieca)*
- [ ] **Task 3:** Identify thermal lag hysteresis loops between heating element power and thermocouple readings *(🇵🇱 Zidentyfikuj pętle histerezy termicznej między mocą grzałek a odczytem termopar)*


In [ ]:
# Exercise Challenge:
# Verify that detected lag between duty cycle and Zone 2 is between 40 and 50 seconds
assert 40 <= optimal_lag <= 50, f"Detected lag {optimal_lag} outside expected 45s physical window."
print("✓ Thermal transport lag calculation verified.")
